# Применение MLP для табличных данных

In [1]:
import pandas as pd
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="matfu21/yambda-50m-lag-features",
    repo_type="dataset",
    filename="listens.parquet",
)

listens = pd.read_parquet(path)

/Users/rshinkarev/Downloads/gp5/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np

def extract_preference_pairs(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["row_id"] = np.arange(len(out))
    out = out.sort_values(["uid", "timestamp", "row_id"]).reset_index(drop=True)

    group = out.groupby("uid", sort=False)

    prev_like = group["is_like"].shift(1)
    next_like = group["is_like"].shift(-1)

    prev_full = group["is_full_play"].shift(1)
    next_full = group["is_full_play"].shift(-1)

    diff_prev = (
        prev_like.notna()
        & (
            out["is_like"].ne(prev_like)
            | out["is_full_play"].ne(prev_full)
        )
    )

    diff_next = (
        next_like.notna()
        & (
            out["is_like"].ne(next_like)
            | out["is_full_play"].ne(next_full)
        )
    )

    return (
        out[diff_prev | diff_next]
        .drop(columns=["row_id"])
        .reset_index(drop=True)
    )

listens = extract_preference_pairs(listens)


In [3]:
from datasets import DatasetDict, load_dataset


def load_precomputed_listens() -> pd.DataFrame:
    path = hf_hub_download(
        repo_id="matfu21/yambda-50m-lag-features",
        repo_type="dataset",
        filename="listens.parquet",
    )
    return pd.read_parquet(path)


def load_yambda_table(filename: str) -> pd.DataFrame:
    data = load_dataset(
        "yandex/yambda",
        data_dir="",
        data_files=f"{filename}.parquet",
    )

    assert isinstance(data, DatasetDict)
    return data["train"].to_pandas()

listens = load_precomputed_listens()

albums = load_yambda_table("album_item_mapping")
artists = load_yambda_table("artist_item_mapping")


In [4]:
import gc


def build_list_mapping(
    df: pd.DataFrame,
    key_col: str,
    value_col: str,
) -> dict:
    return (
        df
        .drop_duplicates([key_col, value_col])
        .sort_values([key_col, value_col])
        .groupby(key_col)[value_col]
        .agg(list)
        .to_dict()
    )


def join_item_artist_album(
    listens: pd.DataFrame,
    artists: pd.DataFrame,
    albums: pd.DataFrame,
) -> pd.DataFrame:
    out = listens.copy()

    artist_map = build_list_mapping(artists, "item_id", "artist_id")
    album_map = build_list_mapping(albums, "item_id", "album_id")

    out["artist_ids"] = out["item_id"].map(artist_map)
    out["album_ids"] = out["item_id"].map(album_map)

    out["artist_ids"] = [x if isinstance(x, list) else [] for x in out["artist_ids"]]
    out["album_ids"] = [x if isinstance(x, list) else [] for x in out["album_ids"]]

    return out

listens = join_item_artist_album(listens, artists, albums)

del artists, albums
gc.collect()

0

In [5]:
def temporal_train_test_split(
    df: pd.DataFrame,
    test_last_seconds: float,
    time_column: str = "timestamp",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    max_time = df[time_column].max()
    split_time = max_time - test_last_seconds

    train = df[df[time_column] < split_time].copy()
    test = df[df[time_column] >= split_time].copy()

    return train.reset_index(drop=True), test.reset_index(drop=True)

train_listens, test_listens = temporal_train_test_split(listens, test_last_seconds=30 * 24 * 60 * 60)

del listens
gc.collect()

0

# MLP 

In [6]:
import gc
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

In [7]:
DENSE_COLUMNS = (
    "user_lag_listen_cnt",
    "user_lag_like_cnt",
    "user_lag_full_play_cnt",
    "user_lag_skip_cnt",
    "item_lag_listen_cnt",
    "item_lag_like_cnt",
    "item_lag_full_play_cnt",
    "item_lag_skip_cnt",
    "ui_lag_listen_cnt",
    "ui_lag_like_cnt",
    "ui_lag_full_play_cnt",
    "ui_lag_skip_cnt",
    "user_lag_avg_played_ratio",
    "item_lag_avg_played_ratio",
    "ui_lag_avg_played_ratio",
)

SPARSE_COLUMNS = ("uid_hash", "item_id_hash")

MULTIVALENT_COLUMNS = ("artist_hash_ids", "album_hash_ids")

LABEL_COLUMNS = ("is_like", "is_full_play")

In [8]:
def hash_id_list(xs, cardinality: int):
    if not isinstance(xs, list):
        return []

    return [int(x) % cardinality for x in xs]


def add_hashed_features(df: pd.DataFrame, cardinality: int) -> pd.DataFrame:
    df = df.copy()

    df["uid_hash"] = df["uid"].astype("int64") % cardinality
    df["item_id_hash"] = df["item_id"].astype("int64") % cardinality

    df["artist_hash_ids"] = df["artist_ids"].apply(
        lambda xs: hash_id_list(xs, cardinality)
    )

    df["album_hash_ids"] = df["album_ids"].apply(
        lambda xs: hash_id_list(xs, cardinality)
    )

    return df

In [9]:
class RankerDataset(Dataset):
    def __init__(self,
        df: pd.DataFrame,
        label_columns: list[str],
        dense_columns: list[str],
        sparse_columns: list[str],
        multivalent_columns: list[str],
        batch_size: int,
    ):
        if batch_size < 1:
            raise ValueError("batch_size must be >= 1")

        self.batches = []

        for start in range(0, len(df), batch_size):
            batch_df = df.iloc[start:start + batch_size]

            batch = {
                "labels": {
                    col: torch.tensor(
                        batch_df[col].to_numpy(),
                        dtype=torch.float32,
                    )
                    for col in label_columns
                },
                "dense_features": torch.tensor(
                    batch_df[list(dense_columns)].to_numpy(dtype=np.float32),
                    dtype=torch.float32,
                ),
                "sparse_features": {
                    col: torch.tensor(
                        batch_df[col].to_numpy(),
                        dtype=torch.long,
                    )
                    for col in sparse_columns
                },
                "multivalent_features": self._make_multivalent_features(
                    batch_df,
                    multivalent_columns,
                ),
                "meta": {
                    "timestamp": torch.tensor(
                        batch_df["timestamp"].to_numpy(),
                        dtype=torch.long,
                    ),
                    "uid": torch.tensor(
                        batch_df["uid"].to_numpy(),
                        dtype=torch.long,
                    ),
                    "item_id": torch.tensor(
                        batch_df["item_id"].to_numpy(),
                        dtype=torch.long,
                    ),
                },
            }

            self.batches.append(batch)

    @staticmethod
    def _make_multivalent_features(
        batch_df: pd.DataFrame,
        multivalent_columns: list[str],
    ) -> dict[str, dict[str, torch.Tensor]]:
        result = {}

        for col in multivalent_columns:
            lists = batch_df[col].tolist()

            lengths = [
                len(x) if isinstance(x, list) else 0
                for x in lists
            ]

            values = []

            for x in lists:
                if isinstance(x, list):
                    values.extend(x)

            result[col] = {
                "values": torch.tensor(values, dtype=torch.long),
                "lengths": torch.tensor(lengths, dtype=torch.long),
            }

        return result

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, idx):
        return self.batches[idx]

In [10]:
class PiecewiseLinearEncoder(nn.Module):
    @staticmethod
    def compute_bins(
        X: torch.Tensor,
        n_bins: int,
    ) -> list[torch.Tensor]:
        quantiles = torch.linspace(
            0.0,
            1.0,
            n_bins + 1,
            device=X.device,
            dtype=X.dtype,
        )

        bins = []

        for feature_idx in range(X.shape[1]):
            feature_bins = torch.quantile(X[:, feature_idx], quantiles)
            feature_bins = torch.unique(feature_bins, sorted=True)

            n_feature_bins = len(feature_bins) - 1
            assert n_feature_bins >= 1, "There is a column with only one unique value"

            bins.append(feature_bins)

        return bins

    @classmethod
    def from_dataset(
        cls,
        dense_train_df,
        n_bins: int = 32,
        train_df_slice: int = 1_000_000,
    ):
        if isinstance(dense_train_df, pd.DataFrame):
            X = torch.tensor(
                dense_train_df.iloc[:train_df_slice].to_numpy(dtype=np.float32),
                dtype=torch.float32,
            )
        elif isinstance(dense_train_df, torch.Tensor):
            X = dense_train_df[:train_df_slice].to(torch.float32)
        else:
            X = torch.tensor(
                dense_train_df[:train_df_slice].to_numpy(dtype=np.float32),
                dtype=torch.float32,
            )

        bins = cls.compute_bins(X, n_bins)

        n_bins_per_feature = [len(b) - 1 for b in bins]
        max_n_bins = max(n_bins_per_feature)
        n_features = len(bins)

        weight = torch.zeros(n_features, max_n_bins, dtype=torch.float32)
        bias = torch.zeros(n_features, max_n_bins, dtype=torch.float32)

        for feature_idx, feature_bins in enumerate(bins):
            left = feature_bins[:-1]
            right = feature_bins[1:]

            w = 1.0 / (right - left)
            b = -left / (right - left)

            cur_n_bins = len(feature_bins) - 1
            weight[feature_idx, :cur_n_bins] = w
            bias[feature_idx, :cur_n_bins] = b

        if len(set(n_bins_per_feature)) == 1:
            mask = None
        else:
            mask = torch.zeros(n_features, max_n_bins, dtype=torch.bool)

            for feature_idx, cur_n_bins in enumerate(n_bins_per_feature):
                mask[feature_idx, :cur_n_bins] = True

            mask = mask.flatten()

        single_bin_mask = torch.tensor(
            [x == 1 for x in n_bins_per_feature],
            dtype=torch.bool,
        )

        if not single_bin_mask.any():
            single_bin_mask = None

        return cls(
            weight=weight,
            bias=bias,
            mask=mask,
            n_bins=n_bins_per_feature,
            single_bin_mask=single_bin_mask,
        )

    def __init__(
        self,
        weight,
        bias,
        mask,
        n_bins,
        single_bin_mask,
    ):
        super().__init__()

        self._n_bins = list(n_bins)

        self.register_buffer("weight", weight)
        self.register_buffer("bias", bias)

        if mask is None:
            self.mask = None
        else:
            self.register_buffer("mask", mask)

        if single_bin_mask is None:
            self.single_bin_mask = None
        else:
            self.register_buffer("single_bin_mask", single_bin_mask)

    @property
    def n_bins(self):
        return self._n_bins

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(self.weight.dtype)

        out = self.bias.unsqueeze(0) + self.weight.unsqueeze(0) * x.unsqueeze(-1)

        if out.shape[-1] == 1:
            out = out.clamp(0.0, 1.0)
        else:
            first = out[..., :1].clamp_max(1.0)
            middle = out[..., 1:-1].clamp(0.0, 1.0)
            last = out[..., -1:].clamp_min(0.0)

            out = torch.cat([first, middle, last], dim=-1)

            if self.single_bin_mask is not None:
                out[:, self.single_bin_mask, :] = out[
                    :, self.single_bin_mask, :
                ].clamp(0.0, 1.0)

        out = out.flatten(start_dim=1)

        if self.mask is not None:
            out = out[:, self.mask]

        return out

In [11]:
class ConcatMLP(nn.Module):
    def __init__(
        self,
        embedding_size: int,
        deep_units: list[int],
        dense_train_df: pd.DataFrame,
        n_bins: int,
        train_df_slice: int,
        cardinality: int = 65_536,
        output_size: int = 1,
    ):
        super().__init__()

        self.uid_emb = nn.Embedding(cardinality, embedding_size)
        self.item_emb = nn.Embedding(cardinality, embedding_size)

        self.artist_emb = nn.EmbeddingBag(
            cardinality,
            embedding_size,
            mode="mean",
        )

        self.album_emb = nn.EmbeddingBag(
            cardinality,
            embedding_size,
            mode="mean",
        )

        self.dense_encoder = PiecewiseLinearEncoder.from_dataset(
            dense_train_df,
            n_bins=n_bins,
            train_df_slice=train_df_slice,
        )

        input_size = (
            sum(self.dense_encoder.n_bins)
            + 4 * embedding_size
        )

        layers = []
        prev_dim = input_size

        for hidden_dim in deep_units:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            prev_dim = hidden_dim

        self.mlp = nn.Sequential(*layers)
        self.output_layer = nn.Linear(prev_dim, output_size)

    @staticmethod
    def bag_forward(
        embedding_bag: nn.EmbeddingBag,
        values: torch.Tensor,
        lengths: torch.Tensor,
    ) -> torch.Tensor:
        batch_size = lengths.shape[0]

        if values.numel() == 0:
            return torch.zeros(
                batch_size,
                embedding_bag.embedding_dim,
                device=lengths.device,
                dtype=embedding_bag.weight.dtype,
            )

        offsets = torch.cat(
            [
                torch.zeros(1, dtype=torch.long, device=lengths.device),
                lengths.cumsum(dim=0)[:-1],
            ]
        )

        return embedding_bag(values, offsets)

    def forward(self, inputs: dict) -> torch.Tensor:
        dense_features = self.dense_encoder(inputs["dense_features"])

        uid_features = self.uid_emb(
            inputs["sparse_features"]["uid_hash"]
        )

        item_features = self.item_emb(
            inputs["sparse_features"]["item_id_hash"]
        )

        artist_features = self.bag_forward(
            self.artist_emb,
            inputs["multivalent_features"]["artist_hash_ids"]["values"],
            inputs["multivalent_features"]["artist_hash_ids"]["lengths"],
        )

        album_features = self.bag_forward(
            self.album_emb,
            inputs["multivalent_features"]["album_hash_ids"]["values"],
            inputs["multivalent_features"]["album_hash_ids"]["lengths"],
        )

        x = torch.cat(
            [
                dense_features,
                uid_features,
                item_features,
                artist_features,
                album_features,
            ],
            dim=1,
        )

        x = self.mlp(x)
        return self.output_layer(x)

In [12]:
def to_device(x, device):
    if isinstance(x, torch.Tensor):
        return x.to(device)

    if isinstance(x, dict):
        return {
            key: to_device(value, device)
            for key, value in x.items()
        }

    return x

In [13]:
def _pairwise_accuracy(df, score_col, label_col):
    df = df.sort_values(["uid", "timestamp", "item_id"]).reset_index(drop=True)

    same_user = df["uid"].values[1:] == df["uid"].values[:-1]

    y_prev = df[label_col].values[:-1]
    y_next = df[label_col].values[1:]

    s_prev = df[score_col].values[:-1]
    s_next = df[score_col].values[1:]

    diff_mask = same_user & (y_prev != y_next)

    if diff_mask.sum() == 0:
        return 0.0

    y_diff = y_next[diff_mask] - y_prev[diff_mask]
    s_diff = s_next[diff_mask] - s_prev[diff_mask]

    return float((y_diff * s_diff > 0).mean())


@torch.no_grad()
def evaluate_pairwise(model, test_loader):
    model.eval()

    device = next(model.parameters()).device

    uid_list = []
    item_id_list = []
    timestamp_list = []
    like_list = []
    full_play_list = []
    score_full_play_list = []
    score_like_list = []

    for batch in test_loader:
        batch = to_device(batch, device)

        logits = model(batch)

        if logits.ndim == 1:
            logits = logits.unsqueeze(1)

        logits = logits.detach().cpu()

        uid_list.append(batch["meta"]["uid"].detach().cpu().numpy())
        item_id_list.append(batch["meta"]["item_id"].detach().cpu().numpy())
        timestamp_list.append(batch["meta"]["timestamp"].detach().cpu().numpy())
        like_list.append(batch["labels"]["is_like"].detach().cpu().numpy())
        full_play_list.append(batch["labels"]["is_full_play"].detach().cpu().numpy())

        if logits.shape[1] == 1:
            score_full_play_list.append(logits[:, 0].numpy())
        else:
            score_like_list.append(logits[:, 0].numpy())
            score_full_play_list.append(logits[:, 1].numpy())

    pred_df = pd.DataFrame(
        {
            "uid": np.concatenate(uid_list),
            "item_id": np.concatenate(item_id_list),
            "timestamp": np.concatenate(timestamp_list),
            "is_like": np.concatenate(like_list),
            "is_full_play": np.concatenate(full_play_list),
            "score_full_play": np.concatenate(score_full_play_list),
        }
    )

    metrics = {
        "pair_accuracy_full_play": _pairwise_accuracy(
            pred_df,
            score_col="score_full_play",
            label_col="is_full_play",
        ),
        "pair_accuracy_full_play_like_pairs": _pairwise_accuracy(
            pred_df,
            score_col="score_full_play",
            label_col="is_like",
        ),
    }

    if len(score_like_list) > 0:
        pred_df["score_like"] = np.concatenate(score_like_list)

        metrics["pair_accuracy_like"] = _pairwise_accuracy(
            pred_df,
            score_col="score_like",
            label_col="is_like",
        )

        metrics["pair_accuracy_like_full_play_pairs"] = _pairwise_accuracy(
            pred_df,
            score_col="score_like",
            label_col="is_full_play",
        )

    return metrics

In [14]:
def train_model(
    model,
    train_loader,
    test_loader,
    epochs,
    lr,
    train_log_every,
):
    device = next(model.parameters()).device

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    last_metrics = None

    for epoch in range(1, epochs + 1):
        model.train()

        losses = []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")

        for step, batch in enumerate(pbar, start=1):
            batch = to_device(batch, device)

            optimizer.zero_grad()

            logits = model(batch)

            if logits.ndim == 1:
                logits = logits.unsqueeze(1)

            if logits.shape[1] == 1:
                target = batch["labels"]["is_full_play"].unsqueeze(1)
            elif logits.shape[1] == 2:
                target = torch.stack(
                    [
                        batch["labels"]["is_like"],
                        batch["labels"]["is_full_play"],
                    ],
                    dim=1,
                )
            else:
                raise ValueError(f"Unsupported output_size: {logits.shape[1]}")

            loss = criterion(logits, target.float())

            loss.backward()
            optimizer.step()

            losses.append(loss.item())

            if step % train_log_every == 0:
                pbar.set_postfix(loss=np.mean(losses[-train_log_every:]))

        last_metrics = evaluate_pairwise(model, test_loader)

        print(f"Epoch {epoch}: train_loss={np.mean(losses):.5f}")

        for key, value in last_metrics.items():
            print(f"{key}: {value:.5f}")

    return last_metrics

## Обучение

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

EMBEDDING_SIZE = 16
N_BINS = 32
CARDINALITY = 65_536
BATCH_SIZE = 8192
TRAIN_DF_SLICE = 1_000_000

train_listens_dense_million = (
    train_listens
    .loc[:, list(DENSE_COLUMNS)]
    .iloc[:TRAIN_DF_SLICE]
    .copy()
)

train_listens = add_hashed_features(train_listens, CARDINALITY)
test_listens = add_hashed_features(test_listens, CARDINALITY)

In [16]:
train_dataset = RankerDataset(
    df=train_listens,
    label_columns=list(LABEL_COLUMNS),
    dense_columns=list(DENSE_COLUMNS),
    sparse_columns=list(SPARSE_COLUMNS),
    multivalent_columns=list(MULTIVALENT_COLUMNS),
    batch_size=BATCH_SIZE,
)

test_dataset = RankerDataset(
    df=test_listens,
    label_columns=list(LABEL_COLUMNS),
    dense_columns=list(DENSE_COLUMNS),
    sparse_columns=list(SPARSE_COLUMNS),
    multivalent_columns=list(MULTIVALENT_COLUMNS),
    batch_size=BATCH_SIZE,
)

del train_listens
del test_listens
gc.collect()

0

In [17]:
train_loader = DataLoader(
    train_dataset,
    batch_size=None,
    shuffle=True,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=None,
    shuffle=False,
    num_workers=0,
)

In [18]:
model_concat_mlp = ConcatMLP(
    embedding_size=EMBEDDING_SIZE,
    deep_units=[256, 128],
    dense_train_df=train_listens_dense_million,
    n_bins=N_BINS,
    train_df_slice=TRAIN_DF_SLICE,
    cardinality=CARDINALITY,
    output_size=1,
).to(device)

In [ ]:
metrics = train_model(
    model=model_concat_mlp,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=10,
    lr=1e-3,
    train_log_every=100,
)

metrics["pair_accuracy_full_play"]

Epoch 1/10: 100%|██████████| 2370/2370 [00:34<00:00, 67.79it/s, loss=0.484]


Epoch 1: train_loss=0.48384
pair_accuracy_full_play: 0.56645
pair_accuracy_full_play_like_pairs: 0.47949


Epoch 2/10:  16%|█▌        | 373/2370 [00:05<00:30, 64.99it/s, loss=0.476]

# 4. DCN-v2

В этом задании вам нужно реализовать cross-слой из статьи [DCN V2: Improved Deep & Cross Network and Practical Lessons for Web-scale Learning to Rank Systems](https://arxiv.org/pdf/2008.13535) в варианте **Mixture of Low-Rank Experts**.

Кроме того, дополнительно к `DeepNetwork` вам нужно реализовать ещё две архитектуры:
- `ResDeepNetwork`;
- `DenseDeepNetwork`.

После этого вам нужно будет провести небольшой анализ и сравнить качество разных подходов.

## Mixture Low Rank Cross Layer

Вам нужно реализовать `MixtureLowRankCrossLayer` и `MixtureLowRankCrossNetwork` — low-rank вариант cross-слоя из DCN V2 со смесью экспертов.

Идея этого блока такая: входной вектор признаков несколько раз пропускается через специальные cross-слои, которые моделируют явные взаимодействия между признаками. В отличие от обычного полносвязного слоя, здесь новое представление строится через произведение исходного входа `x0` и преобразования текущего состояния `xl`.

В этой реализации каждый cross-слой состоит из смеси low-rank экспертов:
- каждый эксперт задаёт своё low-rank преобразование;
- затем gate-сеть вычисляет веса экспертов;
- итоговое обновление получается как взвешенная сумма выходов всех экспертов.

### Что нужно реализовать в `MixtureLowRankCrossLayer`

В конструкторе нужно:
1. сохранить `input_dim`, `num_experts` и `rank`;
2. создать обучаемые параметры:
   - `U` формы `[num_experts, input_dim, rank]`,
   - `V` формы `[num_experts, input_dim, rank]`,
   - `bias` формы `[input_dim]`;
3. создать `gate` — линейный слой, который по `xl` предсказывает веса экспертов;
4. инициализировать параметры.

В `forward(x0, xl)` нужно:
1. применить low-rank преобразование к `xl` через `V`, а затем через `U`;
2. прибавить `bias`;
3. умножить результат поэлементно на `x0`;
4. посчитать веса экспертов через `softmax(gate(xl))`;
5. смешать выходы экспертов с этими весами;
6. прибавить residual `xl`.

Итоговый выход должен иметь ту же форму, что и вход:
```python
[batch_size, input_dim]
```

Что нужно реализовать в `MixtureLowRankCrossNetwork`

Это просто стек из нескольких MixtureLowRankCrossLayer.

В конструкторе нужно:
1. создать num_layers cross-слоёв;
2. сохранить их в nn.ModuleList.

В forward(x) нужно:
1. сохранить исходный вход как x0;
2. завести текущее состояние xl = x;
3. последовательно прогнать xl через все cross-слои;
4. вернуть результат последнего слоя.

Важная деталь

Во всей сети x0 остаётся фиксированным и всегда равен исходному входу, а xl меняется от слоя к слою. 

Ожидаемое поведение
- MixtureLowRankCrossLayer принимает x0 и xl формы [batch_size, input_dim] и возвращает тензор той же формы.
- MixtureLowRankCrossNetwork принимает x формы [batch_size, input_dim] и возвращает тензор той же формы.



In [ ]:
class MixtureLowRankCrossLayer(nn.Module):
    def __init__(self, input_dim: int, num_experts: int, rank: int):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, x0: torch.Tensor, xl: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


class MixtureLowRankCrossNetwork(nn.Module):
    def __init__(self, input_dim: int, num_layers: int, num_experts: int, rank: int):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


tests.test_mixture_low_rank_cross_network(MixtureLowRankCrossLayer, MixtureLowRankCrossNetwork)

## ResDeepNetwork

Теперь вам нужно реализовать `ResidualMLPBlock` и `ResDeepNetwork` — residual-вариант глубокой полносвязной сети.

Идея здесь такая: вместо обычной последовательности `Linear -> ReLU -> Linear -> ReLU` каждый блок дополнительно использует skip connection. Это помогает стабилизировать обучение и облегчает прохождение градиентов через глубокую сеть.

### ResidualMLPBlock

`ResidualMLPBlock` — это residual-блок из двух линейных слоёв.

В конструкторе нужно:
- создать первый линейный слой;
- создать второй линейный слой;
- при необходимости добавить проекцию `proj`, если размер входа `in_dim` не совпадает с размером выхода `out_dim`.

В `forward` нужно:
1. пропустить вход через MLP-ветку;
2. прибавить skip connection;
3. если размеры не совпадают, сначала применить `proj` к входу;
4. после сложения применить `ReLU`.

Если `in_dim == out_dim`, то в skip connection можно использовать вход `x` напрямую. Если размеры отличаются, нужно сначала перевести вход в размерность `out_dim` через линейную проекцию.

### ResDeepNetwork

`ResDeepNetwork` — это последовательность residual-блоков.

В конструкторе передаются:
- `input_dim` — размер входа;
- `hidden_units` — список выходных размерностей residual-блоков.

Нужно построить цепочку блоков так, чтобы каждый следующий блок принимал выход предыдущего. Для этого можно завести размеры

```python
[input_dim] + hidden_units
```

и затем создать блоки между соседними размерностями.

В `forward` нужно просто последовательно прогнать вход через все блоки.

Ожидаемое поведение

Если `hidden_units = [h1, h2, ..., hk]`, то для входа формы

```python
[batch_size, input_dim]
```

выход ResDeepNetwork должен иметь форму

```python
[batch_size, hk]
```

   

In [ ]:
class ResidualMLPBlock(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


class ResDeepNetwork(nn.Module):
    def __init__(self, input_dim: int, hidden_units: list[int]):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


tests.test_res_deep_network(ResidualMLPBlock, ResDeepNetwork)

## DenseDeepNetwork

Теперь вам нужно реализовать `DenseDeepNetwork` — полносвязную сеть с dense connectivity.

Идея этой архитектуры такая: каждый следующий слой получает на вход не только выход предыдущего слоя, но и все предыдущие представления, включая исходный вход. То есть на каждом шаге вход в слой строится как конкатенация

```python
x, h_1, h_2, ..., h_{i-1}
```

Это позволяет каждому слою напрямую использовать более ранние признаки и уже посчитанные представления.

При этом важно: конкатенация используется только для формирования входа в следующий слой. Выход самого очередного слоя — это обычный тензор размера, заданного в hidden_units, а не накопленная конкатенация всех прошлых выходов.

Что нужно сделать

В конструкторе передаются:
- input_dim — размер исходного входа;
- hidden_units — список размерностей скрытых слоёв.

Нужно:
1. создать последовательность линейных слоёв;
2. учесть, что размер входа в каждый следующий слой растёт, потому что к нему конкатенируются все предыдущие выходы.

В forward нужно:
1. завести список features, который сначала содержит только исходный вход x;
2. на каждом шаге сконкатенировать все тензоры из features;
3. применить очередной линейный слой и ReLU;
4. добавить новый выход в features;
5. вернуть выход последнего слоя.

Ожидаемое поведение

Если `hidden_units = [h1, h2, ..., hk]`, то для входа формы

```python
[batch_size, input_dim]
```

выход должен иметь форму

```pythonj
[batch_size, hk]
```

In [ ]:
class DenseDeepNetwork(nn.Module):
    def __init__(self, input_dim: int, hidden_units: list[int]):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


tests.test_dense_deep_network(DenseDeepNetwork)

## DCN V2

Далее вам нужно реализовать итогвую `DCN-v2` модель. Все аналогично `ConcatMLP` модели, но добавляется новый cross-слой, а так же возможность выбрать deep часть. Испольуйте `build_deep_network`.

In [ ]:
def build_deep_network(
    input_dim: int,
    hidden_units: list[int],
    deep_type: str = "mlp",
) -> nn.Module:
    """Factory for the deep tower: ``mlp``, ``resnet``, or ``densenet``."""
    if deep_type == "mlp":
        return DeepNetwork(input_dim, hidden_units)
    if deep_type == "resnet":
        return ResDeepNetwork(input_dim, hidden_units)
    if deep_type == "densenet":
        return DenseDeepNetwork(input_dim, hidden_units)
    raise ValueError(f"Unknown deep_type={deep_type!r}, expected 'mlp', 'resnet', or 'densenet'")


In [ ]:
class DCNV2(nn.Module):
    def __init__(
        self,
        embedding_size,
        cross_layers,
        deep_units,
        input_size,
        dense_train_df,
        n_bins,
        train_df_slice,
        cardinality=65536,
        num_experts: int = 4,
        low_rank: int = 32,
        deep_network: str = "mlp",
        output_size: int = 2,
    ):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


    def forward(self, inputs: dict) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


tests.test_dcnv2(DCNV2)

## Обучение mlp, resnet, densenet с cross-слоями

Ваша задача подобрать параметры для обучения `DCNV2` с `mlp` частью так, чтобы был пройден тест. А так же сравнить этот варинат с `densenet` и `resnet`. 

In [ ]:
model_dcnv2_mlp = DCNV2(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
    deep_network="mlp",
)

metrics = train_model(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
)

tests.test_model_dcnv2_mlp_metrics(metrics)

In [ ]:
model_dcnv2_resnet = DCNV2(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
    deep_network="resnet",
)

metrics = train_model(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
)

In [ ]:
model_dcnv2_resnet = DCNV2(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
    deep_network="densenet",
)

metrics = train_model(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
)

# 5. Многоголовость

Одно из важных преимуществ нейросетей состоит в том, что их можно обучать сразу на несколько таргетов. Например, модель может одновременно решать задачу регрессии (предсказывать число секунд прослушивания) и несколько задач классификации (предсказывать лайк, полное прослушивание и другие типы пользовательского фидбека). На практике таких таргетов может быть довольно много, вплоть до десятков (https://blog.reachsumit.com/posts/2023/04/the-twitter-ml-algo/).

Преимущество такого подхода в том, что модель сразу учится строить общее представление объекта, полезное для разных продуктовых сигналов. После этого уже на этапе анализа или A/B-эксперимента можно подбирать веса разных голов и собирать итоговый скор так, чтобы он лучше соответствовал целям продукта.

В этом задании вы обучите двухголовую нейросетевую модель, которая будет одновременно предсказывать:
- лайк;
- полное прослушивание.

После этого вам нужно будет построить парето-фронт и подобрать такие веса для агрегированного score, чтобы он хорошо работал сразу в двух задачах:
- ранжирование по лайкам;
- ранжирование по полным прослушиваниям.

Если удаётся подобрать хорошие веса, это означает, что итоговое ранжирование одновременно хорошо оптимизирует оба сигнала: пользователи и чаще ставят лайки, и чаще дослушивают треки до конца. Именно такой компромисс обычно и важен для бизнеса.

В конце соберите все результаты в одну таблицу и напишите вывод. 

## Обучение DCNV2 с двумя головами

Вам нужно подобрать параметры обучения так, чтобы пройти тест. В этом задании `output_size=2`, `deep_network='mlp'`.

In [ ]:
model_multitask_dcnv2_mlp = DCNV2(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
    deep_network="mlp",
    output_size=2,
)

metrics = train_model(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
)

tests.test_model_multitask_dcnv2_mlp_metrics(metrics)

## Результаты

| метод | pair_accuracy_like | pair_accuracy_full_play |
|-------|-------------------:|------------------------:|
| random |  |  |
| popular |  |  |
| catboost |  |  |
| concatmlp | |  |
| dcnv2_mlp |  |  |
| dcnv2_resnet |  |  |
| dcnv2_densenet |  |  |
| model_multitask_dcnv2_mlp |  |  |

Вывод: 

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################



## Строим парето-фронт

В этом задании вам нужно построить парето-фронт для двухголовой модели.

В нашем случае парето-фронт — это множество компромиссов между двумя целями:
- качеством ранжирования лайков;
- качеством ранжирования полных прослушиваний.

Идея такая: у модели есть две головы, одна предсказывает вероятность лайка, а другая — вероятность полного прослушивания. Из этих двух предсказаний можно собрать итоговый score как взвешенную сумму

$$
s = \alpha \cdot P(\text{like}) + (1 - \alpha) \cdot P(\text{full\_play}),
$$

где $\alpha \in [0, 1]$.

Важно, что в реальной системе ранжирование всё равно обычно происходит **по одному итоговому score**. Даже если модель предсказывает сразу несколько сигналов, на этапе выдачи объектов нужно отсортировать их по одному числу. Поэтому после обучения многоголовой модели нужно понять, как именно агрегировать выходы разных голов в единый score. Как раз для этого и строится парето-фронт.

При разных значениях $\alpha$ мы получаем разные итоговые ранжирования, а значит и разные значения метрик:
- `pair_accuracy_like`;
- `pair_accuracy_full_play`.

От вас требуется:
1. перебрать разные значения $\alpha$;
2. для каждого значения посчитать итоговый score;
3. вычислить `pair_accuracy_like` и `pair_accuracy_full_play`;
4. построить scatter plot:
   - по оси OX — `pair_accuracy_like`;
   - по оси OY — `pair_accuracy_full_play`.

После этого посмотрите на получившийся график и напишите, какие выводы можно сделать. Используйте хотя бы 100 точек. Возьмите модель из предыдущего пункта. 

In [ ]:
import matplotlib.pyplot as plt

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

Выводы по графику: 

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

## Подберите итоговый score

Теперь вам нужно подобрать такое значение $\alpha$, при котором итоговый score удовлетворяет ограничениям в тесте.

Напомним, что итоговый score строится как

$$
s = \alpha \cdot P(\text{like}) + (1 - \alpha) \cdot P(\text{full\_play}).
$$

На это можно смотреть как на выбор итогового продуктового компромисса: насколько система готова разменивать качество по лайкам на качество по полным прослушиваниям. В реальной задаче именно такой выбор и приходится делать при построении финального ранжирования.

В этом задании вам нужно:
1. подобрать значение $\alpha$;
2. посчитать итоговый score с этим весом;
3. убедиться, что он проходит ограничения из теста.

In [ ]:
alpha =

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    ),
    "pair_accuracy_full_play": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    ),
}

tests.test_model_multitask_dcnv2_mlp_combined_score_metrics(metrics)